In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 

In [2]:
customers = pd.read_csv('../Datasets/archive/olist_customers_dataset.csv')
orders = pd.read_csv('../Datasets/archive/olist_orders_dataset.csv')
items = pd.read_csv('../Datasets/archive/olist_order_items_dataset.csv')

In [3]:
customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [4]:
customers.shape

(99441, 5)

In [5]:
customers.isnull().sum()

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

In [6]:
customers[['customer_id','customer_unique_id']].nunique()

customer_id           99441
customer_unique_id    96096
dtype: int64

In [7]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [8]:
orders.shape

(99441, 8)

In [9]:
customers_orders = customers[['customer_id','customer_unique_id']].merge(orders[['customer_id','order_id','order_status','order_purchase_timestamp']],on='customer_id',how='left')

In [10]:
customers_orders.head()

,customer_id,customer_unique_id,order_id,order_status,order_purchase_timestamp
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,00e7ee1b050b8499577073aeb2a297a1,delivered,2017-05-16 15:05:35
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,29150127e6685892b6eab3eec79f59c7,delivered,2018-01-12 20:48:24
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,b2059ed67ce144a36e2aa97d2c9e9ad2,delivered,2018-05-19 16:07:45
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,951670f92359f4fe4a63112aa7306eba,delivered,2018-03-13 16:06:38
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,6b7d50bd145f6fc7f33cebabd7e49d0f,delivered,2018-07-29 09:51:30


In [11]:
customers_orders.shape

(99441, 5)

In [12]:
customers_orders.isnull().sum()

customer_id                 0
customer_unique_id          0
order_id                    0
order_status                0
order_purchase_timestamp    0
dtype: int64

In [13]:
customers_orders[['customer_unique_id','order_id']].nunique()

customer_unique_id    96096
order_id              99441
dtype: int64

In [14]:
customers_orders.groupby(pd.to_datetime(customers_orders['order_purchase_timestamp']).dt.to_period('M')).agg({"customer_unique_id":"nunique"})

,customer_unique_id
order_purchase_timestamp,
2016-09,4
2016-10,321
2016-12,1
2017-01,765
2017-02,1755
2017-03,2642
2017-04,2372
2017-05,3625
2017-06,3180


In [15]:
customers_orders.groupby(pd.to_datetime(customers_orders['order_purchase_timestamp']).dt.to_period('Y')).agg({"customer_unique_id":"nunique"})

,customer_unique_id
order_purchase_timestamp,
2016,326
2017,43713
2018,52749


In [16]:
first_order = (customers_orders.groupby('customer_unique_id')['order_purchase_timestamp'].min().reset_index())

In [17]:
first_order['acquisition_month'] = (pd.to_datetime(first_order['order_purchase_timestamp']).dt.to_period("M"))

In [18]:
first_order

,customer_unique_id,order_purchase_timestamp,acquisition_month
0,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10 10:56:27,2018-05
1,0000b849f77a49e4a4ce2b2a4ca5be3f,2018-05-07 11:11:27,2018-05
2,0000f46a3911fa3c0805444483337064,2017-03-10 21:05:03,2017-03
3,0000f6ccb0745a6a4b88665a16c9f078,2017-10-12 20:29:41,2017-10
4,0004aac84e0df4da2b147fca70cf8255,2017-11-14 19:45:42,2017-11
...,...,...,...
96091,fffcf5a5ff07b0908bd4e2dbc735a684,2017-06-08 21:00:36,2017-06
96092,fffea47cd6d3cc0a88bd621562a9d061,2017-12-10 20:07:56,2017-12
96093,ffff371b4d645b6ecea244b27531430a,2017-02-07 15:49:16,2017-02
96094,ffff5962728ec6157033ef9805bacc48,2018-05-02 15:17:41,2018-05


In [19]:
monthly_new_customer = (first_order.groupby('acquisition_month').size().reset_index(name='new_customers'))

In [20]:
monthly_new_customer.head()

,acquisition_month,new_customers
0,2016-09,4
1,2016-10,321
2,2016-12,1
3,2017-01,764
4,2017-02,1752


In [21]:
customers_orders['order_month'] = pd.to_datetime(customers_orders['order_purchase_timestamp']).dt.to_period('M')

In [22]:
customers_orders.groupby('order_month')['customer_unique_id'].nunique()

order_month
2016-09       4
2016-10     321
2016-12       1
2017-01     765
2017-02    1755
2017-03    2642
2017-04    2372
2017-05    3625
2017-06    3180
2017-07    3947
2017-08    4246
2017-09    4212
2017-10    4561
2017-11    7430
2017-12    5603
2018-01    7166
2018-02    6569
2018-03    7115
2018-04    6882
2018-05    6814
2018-06    6128
2018-07    6230
2018-08    6460
2018-09      14
2018-10       4
Freq: M, Name: customer_unique_id, dtype: int64

In [23]:
customers_orders.head()

,customer_id,customer_unique_id,order_id,order_status,order_purchase_timestamp,order_month
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,00e7ee1b050b8499577073aeb2a297a1,delivered,2017-05-16 15:05:35,2017-05
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,29150127e6685892b6eab3eec79f59c7,delivered,2018-01-12 20:48:24,2018-01
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,b2059ed67ce144a36e2aa97d2c9e9ad2,delivered,2018-05-19 16:07:45,2018-05
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,951670f92359f4fe4a63112aa7306eba,delivered,2018-03-13 16:06:38,2018-03
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,6b7d50bd145f6fc7f33cebabd7e49d0f,delivered,2018-07-29 09:51:30,2018-07


In [24]:
customers_orders_count = customers_orders.groupby('customer_unique_id')['customer_id'].count()

In [25]:
customers_orders_count = customers_orders_count.reset_index(name="order_count")

In [26]:
customers_orders_count["customers_type"] = "one-time customer"

In [27]:
customers_orders.head()

,customer_id,customer_unique_id,order_id,order_status,order_purchase_timestamp,order_month
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,00e7ee1b050b8499577073aeb2a297a1,delivered,2017-05-16 15:05:35,2017-05
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,29150127e6685892b6eab3eec79f59c7,delivered,2018-01-12 20:48:24,2018-01
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,b2059ed67ce144a36e2aa97d2c9e9ad2,delivered,2018-05-19 16:07:45,2018-05
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,951670f92359f4fe4a63112aa7306eba,delivered,2018-03-13 16:06:38,2018-03
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,6b7d50bd145f6fc7f33cebabd7e49d0f,delivered,2018-07-29 09:51:30,2018-07


In [28]:
customers_orders_count.loc[(customers_orders_count['order_count']>1),"customers_type"] = "Repeat customer"

In [29]:
customers_orders_count['customers_type'].value_counts()

customers_type
one-time customer    93099
Repeat customer       2997
Name: count, dtype: int64

In [30]:
total_unique_customers = customers_orders_count['customer_unique_id'].nunique()
total_unique_customers

96096

In [31]:
one_time_customers = customers_orders_count[customers_orders_count['customers_type'] == 'one-time customer']

In [32]:
repeat_customers = customers_orders_count[customers_orders_count['customers_type'] == 'Repeat customer']

In [33]:
one_time_customers_pct = (one_time_customers['customer_unique_id'].count() / (repeat_customers['customer_unique_id'].count() + one_time_customers['customer_unique_id'].count())*100)

In [34]:
repeat_customers_pct = (repeat_customers['customer_unique_id'].count() / (repeat_customers['customer_unique_id'].count() + one_time_customers['customer_unique_id'].count())*100)

In [35]:
one_time_customers_pct 

np.float64(96.88124375624375)

In [36]:
repeat_customers_pct

np.float64(3.1187562437562435)

In [37]:
customers_orders.head()

,customer_id,customer_unique_id,order_id,order_status,order_purchase_timestamp,order_month
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,00e7ee1b050b8499577073aeb2a297a1,delivered,2017-05-16 15:05:35,2017-05
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,29150127e6685892b6eab3eec79f59c7,delivered,2018-01-12 20:48:24,2018-01
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,b2059ed67ce144a36e2aa97d2c9e9ad2,delivered,2018-05-19 16:07:45,2018-05
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,951670f92359f4fe4a63112aa7306eba,delivered,2018-03-13 16:06:38,2018-03
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,6b7d50bd145f6fc7f33cebabd7e49d0f,delivered,2018-07-29 09:51:30,2018-07


In [38]:
customers_orders_count['customers_type'].value_counts()

customers_type
one-time customer    93099
Repeat customer       2997
Name: count, dtype: int64

In [39]:
customers_orders['order_purchase_timestamp'] = pd.to_datetime(customers_orders['order_purchase_timestamp'])

In [40]:
customers_orders_sorted = customers_orders.sort_values(['customer_unique_id','order_purchase_timestamp'])

In [41]:
repeat_purchase_analysis = (customers_orders_sorted.groupby('customer_unique_id').agg(
    first_orders = ('order_purchase_timestamp','first'), 
    second_orders = ('order_purchase_timestamp',lambda x: x.iloc[1] if len(x)>1 else pd.NaT), 
    order_count = ('order_id','nunique')
).reset_index())

In [42]:
repeat_purchase_analysis.head()

,customer_unique_id,first_orders,second_orders,order_count
0,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10 10:56:27,NaT,1
1,0000b849f77a49e4a4ce2b2a4ca5be3f,2018-05-07 11:11:27,NaT,1
2,0000f46a3911fa3c0805444483337064,2017-03-10 21:05:03,NaT,1
3,0000f6ccb0745a6a4b88665a16c9f078,2017-10-12 20:29:41,NaT,1
4,0004aac84e0df4da2b147fca70cf8255,2017-11-14 19:45:42,NaT,1


In [43]:
repeat_purchase_analysis = repeat_purchase_analysis[repeat_purchase_analysis['order_count']>1].copy()

In [44]:
repeat_purchase_analysis['days_to_second_purchase'] = (
    repeat_purchase_analysis['second_orders']- repeat_purchase_analysis['first_orders']
).dt.total_seconds() / (60 * 60 * 24)

In [45]:
repeat_purchase_analysis['days_to_second_purchase'].describe()

count    2997.000000
mean       80.349981
std       110.192404
min         0.000000
25%         0.001400
50%        27.923773
75%       123.111019
max       608.978912
Name: days_to_second_purchase, dtype: float64

In [46]:
repeat_purchase_analysis['repeat_purchase_timing_distribution'] = "0 - 30 days"

In [47]:
repeat_purchase_analysis.loc[(repeat_purchase_analysis['days_to_second_purchase'] >= 31) & 
    (repeat_purchase_analysis['days_to_second_purchase'] < 91),
    "repeat_purchase_timing_distribution"] = "31 - 90 days"
repeat_purchase_analysis.loc[(repeat_purchase_analysis['days_to_second_purchase'] > 90) & 
    (repeat_purchase_analysis['days_to_second_purchase'] < 181),
    "repeat_purchase_timing_distribution"] = "91 - 180 days" 
repeat_purchase_analysis.loc[(repeat_purchase_analysis['days_to_second_purchase'] > 180),
    "repeat_purchase_timing_distribution"] = "180+ days"

In [48]:
repeat_purchase_analysis['repeat_purchase_timing_distribution'].value_counts()

repeat_purchase_timing_distribution
0 - 30 days      1540
31 - 90 days      518
180+ days         515
91 - 180 days     424
Name: count, dtype: int64

In [49]:
repeat_purchase_analysis.groupby('repeat_purchase_timing_distribution')['order_count'].sum()

repeat_purchase_timing_distribution
0 - 30 days      3264
180+ days        1071
31 - 90 days     1113
91 - 180 days     894
Name: order_count, dtype: int64

In [50]:
repeat_purchase_analysis.head()

,customer_unique_id,first_orders,second_orders,order_count,days_to_second_purchase,repeat_purchase_timing_distribution
33,00172711b30d52eea8b313a7f2cced02,2018-07-28 00:23:49,2018-08-13 09:14:07,2,16.368264,0 - 30 days
106,004288347e5e88a27ded2bb23747066c,2017-07-27 14:13:03,2018-01-14 07:36:54,2,170.724896,91 - 180 days
124,004b45ec5c64187465168251cd1c9c2f,2017-09-01 12:11:23,2018-05-26 19:42:48,2,267.313484,180+ days
144,0058f300f57d7b93c477a131a59b36c3,2018-02-19 17:11:34,2018-03-22 18:09:41,2,31.040359,31 - 90 days
249,00a39521eb40f7012db50455bf083460,2018-05-23 20:14:21,2018-06-03 10:12:57,2,10.582361,0 - 30 days


In [51]:
repeat_purchase_analysis.groupby("repeat_purchase_timing_distribution")['customer_unique_id'].count()

repeat_purchase_timing_distribution
0 - 30 days      1540
180+ days         515
31 - 90 days      518
91 - 180 days     424
Name: customer_unique_id, dtype: int64

In [52]:
customers_orders.head()

,customer_id,customer_unique_id,order_id,order_status,order_purchase_timestamp,order_month
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,00e7ee1b050b8499577073aeb2a297a1,delivered,2017-05-16 15:05:35,2017-05
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,29150127e6685892b6eab3eec79f59c7,delivered,2018-01-12 20:48:24,2018-01
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,b2059ed67ce144a36e2aa97d2c9e9ad2,delivered,2018-05-19 16:07:45,2018-05
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,951670f92359f4fe4a63112aa7306eba,delivered,2018-03-13 16:06:38,2018-03
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,6b7d50bd145f6fc7f33cebabd7e49d0f,delivered,2018-07-29 09:51:30,2018-07


In [53]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [54]:
items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [55]:
customers_orders_items = customers_orders.merge(items[['order_id','product_id','price']],on='order_id')

In [56]:
customers_orders_items.shape

(112650, 8)

In [57]:
customers_orders_items['customer_unique_id'].nunique()

95420

In [58]:
customer_level_analysis = customers_orders_items.groupby("customer_unique_id").agg(
    total_revenue = ("price","sum"), 
    total_orders = ("order_id",'nunique'), 
    total_items = ('product_id','count')
).reset_index()

In [59]:
customer_level_analysis['total_orders'].value_counts().sort_index()

total_orders
1     92507
2      2673
3       192
4        29
5         9
6         5
7         3
9         1
16        1
Name: count, dtype: int64

In [60]:
customer_level_analysis['total_orders'].sum()

np.int64(98666)

In [61]:
missing_customers = set(customers_orders['customer_unique_id']) - set(customers_orders_items['customer_unique_id'])

In [62]:
len(missing_customers)

676

In [63]:
missing_customers_orders = customers_orders[customers_orders['customer_unique_id'].isin(missing_customers)]

In [64]:
missing_customers_orders.head()

,customer_id,customer_unique_id,order_id,order_status,order_purchase_timestamp,order_month
46,f34a6e874087ec1f0e3dab9fdf659c5d,233896de79986082f1f479f1f85281cb,6e98de3a85c84ead6689189b825d35b5,canceled,2018-03-15 10:07:02,2018-03
288,5bfe800011656c0afb81db64519982db,0071f46a072a9ae25bbe4438b15efe9c,df8c077268f7f3baaac0892eb3143642,unavailable,2017-02-01 00:04:17,2017-02
373,b08064e24083fee8fbe8797902b07ecd,035f60af6e7d7f78470e9443be08d339,c609f82bcf7a90292a5940205ebd7e93,unavailable,2018-05-13 16:45:55,2018-05
383,536f46cc0f2f2b1e40d056f7998f0254,340152332a04149987a705602615f0d0,cb4a79c1e6c9ae44302861e7602cc449,unavailable,2017-12-06 15:46:07,2017-12
556,8118922685d2e2c0205f060be4f2579c,d0e87d00021530383c16452a39a393ba,af264f3527e94e431f0dcd56cd6b406d,unavailable,2017-07-12 14:49:11,2017-07


In [65]:
missing_customers_orders['order_id'].nunique()

685

In [66]:
missing_customers_orders['order_status'].value_counts()

order_status
unavailable    569
canceled       109
created          4
invoiced         2
shipped          1
Name: count, dtype: int64

In [67]:
customers_orders_items = customers_orders.merge(items[['order_id','product_id','price']],on='order_id')

In [68]:
customer_level_analysis = customers_orders_items.groupby("customer_unique_id").agg(
    total_revenue = ("price","sum"), 
    total_orders = ("order_id",'nunique'), 
    total_items = ('product_id','count')
).reset_index()

In [69]:
customer_level_analysis['customer_type'] = 'one-time customer'

In [70]:
customer_level_analysis.loc[(customer_level_analysis['total_orders'] > 1),'customer_type'] = "repeat customer"

In [71]:
customer_level_analysis['customer_type'].value_counts()

customer_type
one-time customer    92507
repeat customer       2913
Name: count, dtype: int64

In [72]:
customer_type_analysis = customer_level_analysis.groupby('customer_type').agg(
    total_customers = ('customer_unique_id','nunique'), 
    total_revenue = ('total_revenue','sum'), 
    total_orders = ('total_orders','sum'), 
    average_revenue_per_customer = ('total_revenue','mean'), 
    average_order_per_customer = ('total_orders','mean')
).reset_index()

In [73]:
customer_type_analysis['customers_percentage'] = (customer_type_analysis['total_customers'] / (customer_type_analysis['total_customers'].sum())) * 100
customer_type_analysis['customer_revenue_percentage'] = (customer_type_analysis['total_revenue'] / (customer_type_analysis['total_revenue'].sum())) * 100
customer_type_analysis['customer_orders_percentage'] = (customer_type_analysis['total_orders'] / (customer_type_analysis['total_orders'].sum())) * 100


In [74]:
customer_type_analysis

,customer_type,total_customers,total_revenue,total_orders,average_revenue_per_customer,average_order_per_customer,customers_percentage,customer_revenue_percentage,customer_orders_percentage
0,one-time customer,92507,12828351.84,92507,138.674390,1.000000,96.947181,94.384109,93.757728
1,repeat customer,2913,763291.86,6159,262.029475,2.114315,3.052819,5.615891,6.242272


In [75]:
customer_level_analysis['customer_value_brand'] = pd.qcut(customer_level_analysis['total_revenue'],q=4,
        labels=['low value','medium-low value','medium-high value','high value'])

In [76]:
customer_level_analysis.head()

,customer_unique_id,total_revenue,total_orders,total_items,customer_type,customer_value_brand
0,0000366f3b9a7992bf8c76cfdf3221e2,129.90,1,1,one-time customer,medium-high value
1,0000b849f77a49e4a4ce2b2a4ca5be3f,18.90,1,1,one-time customer,low value
2,0000f46a3911fa3c0805444483337064,69.00,1,1,one-time customer,medium-low value
3,0000f6ccb0745a6a4b88665a16c9f078,25.99,1,1,one-time customer,low value
4,0004aac84e0df4da2b147fca70cf8255,180.00,1,1,one-time customer,high value


In [77]:
customer_level_analysis['customer_value_brand'].value_counts().sort_index()

customer_value_brand
low value            23984
medium-low value     24869
medium-high value    22731
high value           23836
Name: count, dtype: int64

In [78]:
customer_revenue_value_bands = customer_level_analysis.groupby('customer_value_brand').agg(
    total_customers = ('customer_unique_id','count'), 
    total_revenue = ('total_revenue','sum'), 
    total_orders = ("total_orders",'sum'), 
    total_items = ('total_items','sum')
).reset_index()

C:\Users\Admin\AppData\Local\Temp\ipykernel_14900\2948660925.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  customer_revenue_value_bands = customer_level_analysis.groupby('customer_value_brand').agg(


In [79]:
customer_revenue_value_bands

,customer_value_brand,total_customers,total_revenue,total_orders,total_items
0,low value,23984,701218.38,24100,25247
1,medium-low value,24869,1660701.82,25265,27403
2,medium-high value,22731,2716863.83,23452,26805
3,high value,23836,8512859.67,25849,33195


In [80]:
customer_revenue_value_bands['customers_pct'] = (customer_revenue_value_bands['total_customers'] / (customer_revenue_value_bands['total_customers'].sum())) * 100
customer_revenue_value_bands['revenue_pct'] = (customer_revenue_value_bands['total_revenue'] / (customer_revenue_value_bands['total_revenue'].sum())) * 100 
customer_revenue_value_bands['orders_pct'] = (customer_revenue_value_bands['total_orders'] / (customer_revenue_value_bands['total_orders'].sum())) * 100

In [81]:
customer_revenue_value_bands['average_revenue'] = (customer_revenue_value_bands['total_revenue'] / customer_revenue_value_bands['total_customers'])
customer_revenue_value_bands['average_orders'] = (customer_revenue_value_bands['total_orders'] / customer_revenue_value_bands['total_customers'])
customer_revenue_value_bands['average_items'] = (customer_revenue_value_bands['total_items'] / customer_revenue_value_bands['total_customers'])

In [82]:
customer_revenue_value_bands

,customer_value_brand,total_customers,total_revenue,total_orders,total_items,customers_pct,revenue_pct,orders_pct,average_revenue,average_orders,average_items
0,low value,23984,701218.38,24100,25247,25.135192,5.159187,24.425841,29.236924,1.004837,1.052660
1,medium-low value,24869,1660701.82,25265,27403,26.062670,12.218550,25.606592,66.777989,1.015923,1.101894
2,medium-high value,22731,2716863.83,23452,26805,23.822050,19.989222,23.769080,119.522407,1.031719,1.179227
3,high value,23836,8512859.67,25849,33195,24.980088,62.633040,26.198488,357.142963,1.084452,1.392641


In [83]:
customer_level_analysis.head()

,customer_unique_id,total_revenue,total_orders,total_items,customer_type,customer_value_brand
0,0000366f3b9a7992bf8c76cfdf3221e2,129.90,1,1,one-time customer,medium-high value
1,0000b849f77a49e4a4ce2b2a4ca5be3f,18.90,1,1,one-time customer,low value
2,0000f46a3911fa3c0805444483337064,69.00,1,1,one-time customer,medium-low value
3,0000f6ccb0745a6a4b88665a16c9f078,25.99,1,1,one-time customer,low value
4,0004aac84e0df4da2b147fca70cf8255,180.00,1,1,one-time customer,high value


In [84]:
customer_type_analysis

,customer_type,total_customers,total_revenue,total_orders,average_revenue_per_customer,average_order_per_customer,customers_percentage,customer_revenue_percentage,customer_orders_percentage
0,one-time customer,92507,12828351.84,92507,138.674390,1.000000,96.947181,94.384109,93.757728
1,repeat customer,2913,763291.86,6159,262.029475,2.114315,3.052819,5.615891,6.242272


In [85]:
customer_revenue_value_bands

,customer_value_brand,total_customers,total_revenue,total_orders,total_items,customers_pct,revenue_pct,orders_pct,average_revenue,average_orders,average_items
0,low value,23984,701218.38,24100,25247,25.135192,5.159187,24.425841,29.236924,1.004837,1.052660
1,medium-low value,24869,1660701.82,25265,27403,26.062670,12.218550,25.606592,66.777989,1.015923,1.101894
2,medium-high value,22731,2716863.83,23452,26805,23.822050,19.989222,23.769080,119.522407,1.031719,1.179227
3,high value,23836,8512859.67,25849,33195,24.980088,62.633040,26.198488,357.142963,1.084452,1.392641


In [86]:
customer_value_type_analysis = (customer_level_analysis.groupby(['customer_value_brand','customer_type']).agg(
    total_customers = ('customer_unique_id','nunique'), 
    total_revenue = ('total_revenue','sum'), 
    total_orders = ('total_orders','sum'), 
    total_items = ('total_items','sum'), 
    average_revenue = ('total_revenue','mean'), 
    average_orders = ("total_orders",'mean'), 
    average_items = ('total_items','mean')
).reset_index())

C:\Users\Admin\AppData\Local\Temp\ipykernel_14900\2593064985.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  customer_value_type_analysis = (customer_level_analysis.groupby(['customer_value_brand','customer_type']).agg(


In [87]:
customer_value_type_analysis['customer_pct_within_band'] = (customer_value_type_analysis['total_customers'] / (customer_value_type_analysis.groupby('customer_value_brand')['total_customers'].transform('sum')))* 100

C:\Users\Admin\AppData\Local\Temp\ipykernel_14900\144002021.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  customer_value_type_analysis['customer_pct_within_band'] = (customer_value_type_analysis['total_customers'] / (customer_value_type_analysis.groupby('customer_value_brand')['total_customers'].transform('sum')))* 100


In [88]:
customer_value_type_summary = (customer_value_type_analysis.sort_values(
    ['customer_value_brand','customer_type']).reset_index(drop=True))

In [89]:
customer_value_type_analysis

,customer_value_brand,customer_type,total_customers,total_revenue,total_orders,total_items,average_revenue,average_orders,average_items,customer_pct_within_band
0,low value,one-time customer,23873,697287.83,23873,25015,29.208220,1.000000,1.047836,99.537191
1,low value,repeat customer,111,3930.55,227,232,35.410360,2.045045,2.090090,0.462809
2,medium-low value,one-time customer,24488,1633856.91,24488,26569,66.720717,1.000000,1.084980,98.467972
3,medium-low value,repeat customer,381,26844.91,777,834,70.459081,2.039370,2.188976,1.532028
4,medium-high value,one-time customer,22044,2633356.82,22044,25223,119.459119,1.000000,1.144212,96.977696
5,medium-high value,repeat customer,687,83507.01,1408,1582,121.553144,2.049491,2.302766,3.022304
6,high value,one-time customer,22102,7863850.28,22102,28360,355.798130,1.000000,1.283142,92.725289
7,high value,repeat customer,1734,649009.39,3747,4835,374.284539,2.160900,2.788351,7.274711


In [90]:
repeat_customers_by_value_band = customer_value_type_analysis[customer_value_type_analysis['customer_type']=='repeat customer'][['customer_value_brand','customer_pct_within_band','total_customers']]
repeat_customers_by_value_band

,customer_value_brand,customer_pct_within_band,total_customers
1,low value,0.462809,111
3,medium-low value,1.532028,381
5,medium-high value,3.022304,687
7,high value,7.274711,1734


In [91]:
customer_value_type_analysis.groupby(['customer_value_brand','customer_type'])

C:\Users\Admin\AppData\Local\Temp\ipykernel_14900\2155264875.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  customer_value_type_analysis.groupby(['customer_value_brand','customer_type'])


In [92]:
customer_type_analysis

,customer_type,total_customers,total_revenue,total_orders,average_revenue_per_customer,average_order_per_customer,customers_percentage,customer_revenue_percentage,customer_orders_percentage
0,one-time customer,92507,12828351.84,92507,138.674390,1.000000,96.947181,94.384109,93.757728
1,repeat customer,2913,763291.86,6159,262.029475,2.114315,3.052819,5.615891,6.242272


In [93]:
repeat_customers_pct

np.float64(3.1187562437562435)

In [94]:
repeat_customers_by_value_band

,customer_value_brand,customer_pct_within_band,total_customers
1,low value,0.462809,111
3,medium-low value,1.532028,381
5,medium-high value,3.022304,687
7,high value,7.274711,1734


In [95]:
customer_type_analysis

,customer_type,total_customers,total_revenue,total_orders,average_revenue_per_customer,average_order_per_customer,customers_percentage,customer_revenue_percentage,customer_orders_percentage
0,one-time customer,92507,12828351.84,92507,138.674390,1.000000,96.947181,94.384109,93.757728
1,repeat customer,2913,763291.86,6159,262.029475,2.114315,3.052819,5.615891,6.242272


In [96]:
repeat_customers_count = customer_level_analysis[customer_level_analysis['customer_type']=='repeat customer']['customer_unique_id'].nunique()

In [97]:
total_unique_customers = customer_level_analysis['customer_unique_id'].nunique()

In [98]:
repeat_customers_pct = (repeat_customers_count / total_unique_customers) * 100

In [99]:
repeat_customers_pct

3.0528191154894153

In [100]:
repeat_customers_revenue = customer_level_analysis[customer_level_analysis['customer_type']=='repeat customer']['total_revenue'].sum()

In [101]:
total_customers_revenue = customer_level_analysis['total_revenue'].sum()

In [102]:
repeat_customers_revenue_distribution = (repeat_customers_revenue / total_customers_revenue) * 100 
repeat_customers_revenue_distribution

np.float64(5.615890740278896)

In [103]:
repeat_customers_distribution = repeat_customers_by_value_band.copy()

In [104]:
repeat_purchase_analysis

,customer_unique_id,first_orders,second_orders,order_count,days_to_second_purchase,repeat_purchase_timing_distribution
33,00172711b30d52eea8b313a7f2cced02,2018-07-28 00:23:49,2018-08-13 09:14:07,2,16.368264,0 - 30 days
106,004288347e5e88a27ded2bb23747066c,2017-07-27 14:13:03,2018-01-14 07:36:54,2,170.724896,91 - 180 days
124,004b45ec5c64187465168251cd1c9c2f,2017-09-01 12:11:23,2018-05-26 19:42:48,2,267.313484,180+ days
144,0058f300f57d7b93c477a131a59b36c3,2018-02-19 17:11:34,2018-03-22 18:09:41,2,31.040359,31 - 90 days
249,00a39521eb40f7012db50455bf083460,2018-05-23 20:14:21,2018-06-03 10:12:57,2,10.582361,0 - 30 days
...,...,...,...,...,...,...
95784,ff36be26206fffe1eb37afd54c70e18b,2018-07-28 15:49:27,2018-08-20 09:03:23,3,22.718009,0 - 30 days
95810,ff44401d0d8f5b9c54a47374eb48c1b8,2017-05-19 21:20:54,2017-05-19 21:20:54,2,0.000000,0 - 30 days
95916,ff8892f7c26aa0446da53d01b18df463,2017-05-24 16:09:14,2017-11-26 23:25:43,2,186.303113,180+ days
95934,ff922bdd6bafcdf99cb90d7f39cea5b3,2017-02-22 12:26:42,2017-08-23 13:15:29,3,182.033877,180+ days


In [105]:
customer_level_analysis.head()

,customer_unique_id,total_revenue,total_orders,total_items,customer_type,customer_value_brand
0,0000366f3b9a7992bf8c76cfdf3221e2,129.90,1,1,one-time customer,medium-high value
1,0000b849f77a49e4a4ce2b2a4ca5be3f,18.90,1,1,one-time customer,low value
2,0000f46a3911fa3c0805444483337064,69.00,1,1,one-time customer,medium-low value
3,0000f6ccb0745a6a4b88665a16c9f078,25.99,1,1,one-time customer,low value
4,0004aac84e0df4da2b147fca70cf8255,180.00,1,1,one-time customer,high value


In [106]:
customer_type_analysis.head()

,customer_type,total_customers,total_revenue,total_orders,average_revenue_per_customer,average_order_per_customer,customers_percentage,customer_revenue_percentage,customer_orders_percentage
0,one-time customer,92507,12828351.84,92507,138.674390,1.000000,96.947181,94.384109,93.757728
1,repeat customer,2913,763291.86,6159,262.029475,2.114315,3.052819,5.615891,6.242272


In [108]:
customer_value_type_summary

,customer_value_brand,customer_type,total_customers,total_revenue,total_orders,total_items,average_revenue,average_orders,average_items,customer_pct_within_band
0,low value,one-time customer,23873,697287.83,23873,25015,29.208220,1.000000,1.047836,99.537191
1,low value,repeat customer,111,3930.55,227,232,35.410360,2.045045,2.090090,0.462809
2,medium-low value,one-time customer,24488,1633856.91,24488,26569,66.720717,1.000000,1.084980,98.467972
3,medium-low value,repeat customer,381,26844.91,777,834,70.459081,2.039370,2.188976,1.532028
4,medium-high value,one-time customer,22044,2633356.82,22044,25223,119.459119,1.000000,1.144212,96.977696
5,medium-high value,repeat customer,687,83507.01,1408,1582,121.553144,2.049491,2.302766,3.022304
6,high value,one-time customer,22102,7863850.28,22102,28360,355.798130,1.000000,1.283142,92.725289
7,high value,repeat customer,1734,649009.39,3747,4835,374.284539,2.160900,2.788351,7.274711


In [109]:
customer_revenue_value_bands

,customer_value_brand,total_customers,total_revenue,total_orders,total_items,customers_pct,revenue_pct,orders_pct,average_revenue,average_orders,average_items
0,low value,23984,701218.38,24100,25247,25.135192,5.159187,24.425841,29.236924,1.004837,1.052660
1,medium-low value,24869,1660701.82,25265,27403,26.062670,12.218550,25.606592,66.777989,1.015923,1.101894
2,medium-high value,22731,2716863.83,23452,26805,23.822050,19.989222,23.769080,119.522407,1.031719,1.179227
3,high value,23836,8512859.67,25849,33195,24.980088,62.633040,26.198488,357.142963,1.084452,1.392641


In [115]:
repeat_purchase_analysis.shape

(2997, 6)

In [112]:
repeat_customers_by_value_band

,customer_value_brand,customer_pct_within_band,total_customers
1,low value,0.462809,111
3,medium-low value,1.532028,381
5,medium-high value,3.022304,687
7,high value,7.274711,1734


In [116]:
customer_level_analysis.to_csv('customer_level_analysis.csv',index=False) 
customer_value_type_summary.to_csv('customer_value_type_summary.csv',index=False)
customer_revenue_value_bands.to_csv('customer_revenue_value_bands.csv',index=False) 
repeat_customers_by_value_band.to_csv('repeat_customers_by_value_band.csv',index=False)

In [118]:
repeat_customers_distribution

,customer_value_brand,customer_pct_within_band,total_customers
1,low value,0.462809,111
3,medium-low value,1.532028,381
5,medium-high value,3.022304,687
7,high value,7.274711,1734


In [119]:
repeat_customers_revenue

np.float64(763291.86)

In [120]:
repeat_purchase_analysis.head()

,customer_unique_id,first_orders,second_orders,order_count,days_to_second_purchase,repeat_purchase_timing_distribution
33,00172711b30d52eea8b313a7f2cced02,2018-07-28 00:23:49,2018-08-13 09:14:07,2,16.368264,0 - 30 days
106,004288347e5e88a27ded2bb23747066c,2017-07-27 14:13:03,2018-01-14 07:36:54,2,170.724896,91 - 180 days
124,004b45ec5c64187465168251cd1c9c2f,2017-09-01 12:11:23,2018-05-26 19:42:48,2,267.313484,180+ days
144,0058f300f57d7b93c477a131a59b36c3,2018-02-19 17:11:34,2018-03-22 18:09:41,2,31.040359,31 - 90 days
249,00a39521eb40f7012db50455bf083460,2018-05-23 20:14:21,2018-06-03 10:12:57,2,10.582361,0 - 30 days


In [121]:
repeat_purchase_analysis.to_csv('repeat_purchase_analysis.csv',index=False)

# Key Business Findings

## 1. Customer Retention : 
    The customer base is predominately made-up of one-time customers, while repeat customers represent only a small proportion of the analyzed customer base. This indicates a significant opportunity to improve customer retention and encourage second purchases. 

## 2. Customer Value and Repeat behavior : 
    Repeat purchase behavior varies significantly across customer value segments. High-value customers repeat-customer presence, indicating that customer value is positively associated with retention behavior. 

## 3. High value Customer Retention : 
    High-value customers have the highest repeat customer rate among all customer value bands. This segment should be prioritized for retention initiatives and loyalty-focused strategies. 

## 4. One-Time Customer Opportunity : 
    The large number of one-time customers represents a substantial opportunity for customer reactivation and second-purchase conversion. 

## 5. Business Opportunity : 
    The analysis suggests that the business should focus on retaining high-value customers while developing strategies to convert one-time customers into repeat buyers.

# Analytics Outputs :- 
    The following processed datasets were created during the customer analytics workflow : 
        1. customer_level_analysis 
        2. customer_revenue_value_bands 
        3. customer_value_type_summary 
        4. repeat_customers_by_value_band 
        5. repeat_purchase_analysis 
        